# 70. Two untried changes to how the combiner is FIT

**One variable against ledger row 153** (`stack_prune65_last`, CV 0.969962, LB 0.97103): the
combiner's fitting protocol. Same 175 members, same prune to 65, same logistic form, same folds.
Neither change touches membership, and neither is a new model.

Row 150 varied the combiner's FUNCTIONAL FORM and every constrained arm transferred worse. This
varies how the same logistic is fitted, which is a different question and has never been asked here.

## Change one: the test vector wastes a fifth of the data

Every stack row in this repo builds its test prediction as **the average of five combiners, each
fitted on four folds of the out-of-fold matrix**. That protocol exists so the CV number is honest:
no combiner is scored on a row it was fitted on.

**The test set has no such constraint.** No test row appears in the out-of-fold matrix at all, so a
combiner used only to predict test can be fitted on all 691,369 rows. The current arrangement
estimates 65 weights five times on 80 percent of the data and averages them, when it could estimate
them once on 100 percent.

This changes nothing about CV, by construction, and cannot be ranked by it. It is standard practice
and the argument is pure statistics: more rows, better-estimated weights, same estimator.

## Change two: the combiner ignores a measured distribution shift

`05_adversarial_validation.ipynb` measured train against test at **AUC 0.562868** on 2026-08-04, a
mild but real shift, and saved the per-row probabilities to
`artifacts/oof/adversarial_train_prob.npy` with the note that they are there "if fold weighting is
ever tried". **It never was.** The file has sat unused for three weeks.

Importance weighting is the standard response: weight each training row by `p/(1-p)`, where `p` is
its probability of being a test row, so the combiner's weights are fitted toward the part of the
input space the test set actually occupies. On this data those weights span 0.207 to 4.646 with a
p90/p10 ratio of 1.92, so the reweighting is meaningful without being degenerate.

**This should LOWER CV and that is the point.** CV is measured on training rows, which importance
weighting deliberately de-emphasises. The whole competition has been optimising a quantity measured
on the wrong distribution.

## The arms

| arm | OOF fit | test vector |
|---|---|---|
| `A_base` | fold-wise, unweighted | mean of the five fold combiners |
| `B_fulltest` | identical to A | **one combiner fitted on all 691,369 rows** |
| `C_advweight` | fold-wise, **importance weighted** | mean of the five |
| `D_both` | fold-wise, importance weighted | one weighted combiner on all rows |

`A` and `B` share a CV by construction. `C` and `D` share a CV.

## The prediction

**`B` is a small real gain on the leaderboard and invisible on CV**, perhaps +0.00002 to +0.00005,
which at rank 292 with a cutoff seven places away is the right order of magnitude.

**`C` loses CV and I do not know which way it moves the leaderboard.** An adversarial AUC of 0.5629
is genuinely mild, and the repo's own 2026-08-04 entry judged fold weighting "not worth the
complexity at this level". That judgement was made when the stack was one LightGBM.

## The case against

For `B`, the five fold-combiners are fitted on 80 percent overlapping data and their average is
already a low-variance estimate; the gain from the last fifth may be far below what the leaderboard
resolves. Row 24 measured this combiner as barely regularised, `C` from 0.01 to 3.0 moving CV by
2e-6, which says the weights are not variance-limited.

For `C`, an adversarial AUC of 0.5629 means the shift is small, importance weighting is high
variance when weights are noisy, and the estimated weights come from a model fitted three weeks ago
on the raw feature set. If the shift is mostly noise this reweights toward noise.


In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
import json
import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 94's forty-one minus the duplicate, in row 94's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
BASE += [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
         ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw"),
         ("xgb_te_fe", "xgb_te_fe"), ("cat_te_n4000", "cat_te_n4000"),
         ("xgb_raw_fe", "xgb_raw_fe"), ("cat_raw_n10k", "cat_raw_n10000"),
         ("lgb_raw_fe", "lgb_raw_fe"), ("cat_raw_fe", "cat_raw_fe")]
# lgb_raw is deliberately absent: it is bag42 re-run in another kernel, Pearson 0.999970 on
# logits. See the header and the 2026-08-22 entry in NOTES.md. Dropping it costs -0.000001.
DROPPED = [("lgb_raw", "duplicate of bag42, Pearson 0.999970")]

BASE += [("cat_te_fe", "cat_te_fe"), ("hgb_te_fe", "hgb_te_fe"),
         ("lgb_te_fe", "lgb_te_fe"), ("rf_te_fe", "rf_te_fe"),
         ("neural_lookup", "neural_lookup"), ("et_te_fe", "et_te_fe"),
         ("logit_te_fe", "logit_te_fe")]
BASE += [("xgb_tuned", "xgb_tuned")]
BASE += [("neural_fe", "neural_fe"), ("neural_wide", "neural_wide"),
         ("neural_res", "neural_res")]
BASE += [("realmlp", "realmlp")]
BASE += [("realmlp10", "realmlp10")]
BASE += [("realmlp_raw_fe", "realmlp_raw_fe")]
BASE += [("tabm", "tabm")]
# THE ONE VARIABLE. Row 144 held these fifty-five, all built by this repo. The five
# candidates below were not. Each is admitted only by writeup/verify_public_oof.py,
# which proves the fold partition matches ours by reproducing the author's own printed
# per-fold AUCs from our fold vector. See the header.
# Row 145's five, now part of the base.
PRIOR_PUBLIC = ["cb_srcE", "lgb_srcE", "xgb_srcA", "realmlp_srcA", "tabm_srcA",
                "lgb_srcB", "realmlp_srcI", "hgb_srcH", "xgb_srcC", "resnet_kava"]
PUBLIC = PRIOR_PUBLIC
# THE ONE VARIABLE. Thirty-four members from the frozen-fold library, loaded as raw
# .npy rather than through the kernel gate, because their provenance is established by
# transitivity against pub_rmlp and pub_tabm being bit-identical. See the header.
PRIOR_SZY = sorted((ROOT / "artifacts" / "wide_library" / "picked.txt").read_text().split())
_szy_raw = sorted((ROOT / "artifacts" / "wide_library" / "picked2.txt").read_text().split())
PRIOR_SZY2 = [f"szy_{n}" for n in _szy_raw]
# srcK members are single letters, so they are namespaced like the srcL batch.
srcK = [f"gol_{m}" for m in "abcdefg"]
# This batch contains `realmlp` and `xgb_tuned`, which collide with OUR member aliases.
# Namespaced so the loader cannot silently overwrite one of ours.
SZY = []
_SZY_FILE = {f"szy_{n}": n for n in _szy_raw}
# Row 149 holds the srcK seven, row 151 the srcJ twenty-two.
PRIOR_srcK = srcK
import glob as _glob
srcJ = sorted(f"ada_{pathlib.Path(p).name[4:-4]}"
                for p in _glob.glob(str(ROOT / "artifacts" / "catboost_library" / "oof_*.npy")))
PRIOR_srcJ = srcJ
# THE ONE VARIABLE: the last three public sources. Same standing as row 151, a
# documented claim rather than a proof, with the offset as the check. See header.
LAST_SRC = {"srcO": "hbo", "mk_xgb": "mk", "mk_lgb": "mk", "mk_cat": "mk", "srcP": "fm"}

# Selection runs here, BEFORE the member index is built, because a name that is later
# skipped by the loader would otherwise sit in `names` with nothing behind it. Arrays are
# md5-checked against every library file already on disk so a re-publication of a member
# we hold cannot enter twice under a new name.
import hashlib as _hl
_seen_h = set()
for _f in (list((ROOT / "artifacts" / "wide_library").glob("oof_*.npy"))
           + list((ROOT / "artifacts" / "srcK").glob("oof_*.npy"))
           + list((ROOT / "artifacts" / "catboost_library").glob("oof_*.npy"))):
    _v = np.load(_f)
    if _v.shape == (len(train),):
        _seen_h.add(_hl.md5(np.ascontiguousarray(_v.astype(np.float64)).tobytes()).hexdigest())

LAST, LAST_FILES = [], {}
for _folder, _pref in LAST_SRC.items():
    _d = ROOT / "artifacts" / _folder
    for _p in sorted(_d.glob("*oof*.npy")):
        _tp = _d / _p.name.replace("oof", "test", 1)
        if not _tp.exists():
            print(f"  skip {_p.name}: no matching test array")
            continue
        _o, _t = np.load(_p), np.load(_tp)
        _o = _o.ravel() if _o.ndim == 2 and _o.shape[1] == 1 else _o
        _t = _t.ravel() if _t.ndim == 2 and _t.shape[1] == 1 else _t
        if _o.shape != (len(train),) or _t.shape != (len(test),):
            print(f"  skip {_p.name}: shapes {_o.shape} / {_t.shape}")
            continue
        _h = _hl.md5(np.ascontiguousarray(_o.astype(np.float64)).tobytes()).hexdigest()
        if _h in _seen_h:
            print(f"  skip {_p.name}: duplicate of a member already in the pool")
            continue
        _seen_h.add(_h)
        _nm = f"{_pref}_{_p.stem.replace('oof_', '').replace('_oof', '')}"
        LAST.append(_nm)
        LAST_FILES[_nm] = (_p, _tp)
CAND = [(n, n) for n in LAST]
print(f"{len(LAST)} members selected from the last three sources: {LAST}")


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


# Row 145 already holds PRIOR_PUBLIC, so they belong in the index alongside our own.
# Row 147 holds BASE + PRIOR_PUBLIC + PRIOR_SZY, so all three belong in the index.
MEM = (BASE + [(n, n) for n in PRIOR_PUBLIC]
       + [(n, n) for n in PRIOR_SZY + PRIOR_SZY2]
       + [(n, n) for n in PRIOR_srcK] + [(n, n) for n in PRIOR_srcJ] + CAND)
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in BASE}
Ptest = {n: load(s, "test") for n, s in BASE}

# The public five, read from artifacts/public_oof/ with their id order asserted rather
# than assumed. A csv in a different row order blends perfectly cleanly and is invisible
# in the score, which is the failure row 59's loader already guards against.
PUB = ROOT / "artifacts" / "public_oof"
VER = {r["name"]: r for r in json.loads((PUB / "verification.json").read_text(encoding="utf-8"))}


def read_vec(path, n_expected, order_ref):
    """Mirror of writeup/verify_public_oof.load_vector, so a member is loaded here
    exactly as it was loaded when it was verified. Two shapes exist in the wild: a
    csv with or without an id column, and a bare .npy."""
    path = pathlib.Path(path)
    if path.suffix == ".npy":
        v = np.load(path)
        assert len(v) == n_expected, f"{path.name} has {len(v)} rows"
        return v.astype(float)
    df = pd.read_csv(path)
    assert len(df) == n_expected, f"{path.name} has {len(df)} rows"
    idc = [c for c in df.columns if c.lower() == "id"]
    if idc:
        # A csv in a different row order blends perfectly cleanly and is invisible in
        # the score, so this is asserted rather than hoped for.
        assert (df[idc[0]].to_numpy() == order_ref).all(), f"id order {path.name}"
    pref = [c for c in df.columns if any(k in c.lower() for k in ("oof", "pred", "prob"))]
    col = pref or [c for c in df.columns
                   if c.lower() not in ("id", "addicted_label", "target", "fold")]
    col = col or [c for c in df.columns if c.lower() != "id"]
    return df[col[0]].to_numpy(float)


for n in PUBLIC:
    r = VER.get(n)
    assert r and r["verdict"].startswith("ADMISSIBLE"), \
        f"{n} is not admissible: {r['verdict'] if r else 'absent'}. Re-run the gate."
    d = PUB / r["folder"]
    Poof[n] = read_vec(d / r["oof_file"], len(train), train["id"].to_numpy())
    Ptest[n] = read_vec(d / r["test_file"], len(test), test["id"].to_numpy())
# The library members. The manifest AUC is asserted, so a truncated or wrong file
# cannot enter quietly.
import csv as _csv
SZD = ROOT / "artifacts" / "wide_library"
_man = {r["model"]: float(r["oof_auc"])
        for r in _csv.DictReader((SZD / "manifest.csv").open(encoding="utf-8"))}
for n in PRIOR_SZY + PRIOR_SZY2 + SZY:
    stem = _SZY_FILE.get(n, n)
    o = np.load(SZD / f"oof_{stem}.npy")
    t = np.load(SZD / f"test_{stem}.npy")
    assert o.shape == (len(train),) and t.shape == (len(test),), n
    _a = roc_auc_score(y, o)
    assert abs(_a - _man[stem]) < 5e-5, f"{n}: AUC {_a:.6f} vs manifest {_man[stem]}"
    Poof[n], Ptest[n] = o.astype(float), t.astype(float)
print(f"{len(PRIOR_SZY) + len(PRIOR_SZY2)} srcL members carried, all matching manifest AUC")

# The srcK library. Its manifest publishes per-fold AUCs, so the ordinary gate applies:
# every member is PROVEN on our partition rather than accepted on its README.
import csv as _csv2
GOL = ROOT / "artifacts" / "srcK"
for _r in _csv2.DictReader((GOL / "manifest.csv").open(encoding="utf-8")):
    _m = _r["member"]
    _o = np.load(GOL / f"oof_{_m}.npy")
    _t = np.load(GOL / f"test_{_m}.npy")
    assert _o.shape == (len(train),) and _t.shape == (len(test),), _m
    _ours = [roc_auc_score(y[folds == f], _o[folds == f]) for f in range(5)]
    _theirs = [float(_r[f"fold{f}_auc"]) for f in range(5)]
    _d = max(abs(a - b) for a, b in zip(_ours, _theirs))
    assert _d < 1e-4, f"srcK {_m}: fold AUCs differ by {_d:.2e}, a different partition"
    Poof[f"gol_{_m}"], Ptest[f"gol_{_m}"] = _o.astype(float), _t.astype(float)
print(f"{len(srcK)} srcK members verified against their published per-fold AUCs")

# The srcJ library. UNVERIFIED on fold protocol: no per-fold AUCs exist anywhere for
# these. The AUC published in their README is asserted, which catches a corrupt file but
# is partition-independent and proves nothing about the split.
import re as _re
ADA = ROOT / "artifacts" / "catboost_library"
_ada_auc = {m.group(1): float(m.group(2)) for m in _re.finditer(
    r"`oof_([a-z0-9_]+)\.npy`\s*\|\s*(0\.9[0-9]+)", (ADA / "README.md").read_text(encoding="utf-8"))}
for n in srcJ:
    stem = n[4:]
    o = np.load(ADA / f"oof_{stem}.npy")
    t = np.load(ADA / f"test_{stem}.npy")
    assert o.shape == (len(train),) and t.shape == (len(test),), n
    if stem in _ada_auc:
        _a = roc_auc_score(y, o)
        assert abs(_a - _ada_auc[stem]) < 5e-5, f"{n}: AUC {_a:.6f} vs README {_ada_auc[stem]}"
    Poof[n], Ptest[n] = o.astype(float), t.astype(float)
print(f"{len(srcJ)} srcJ members loaded, AUC-checked, FOLD PROTOCOL UNVERIFIED")

# The last three sources, loaded from the file map resolved above.
for _nm, (_p, _tp) in LAST_FILES.items():
    _o, _t = np.load(_p), np.load(_tp)
    Poof[_nm] = (_o.ravel() if _o.ndim == 2 else _o).astype(float)
    Ptest[_nm] = (_t.ravel() if _t.ndim == 2 else _t).astype(float)
print(f"{len(LAST)} members loaded from srcO, srcQ and srcP")
print("  fold protocol DOCUMENTED, NOT PROVEN. The offset on the submission is the check.")
print(f"{len(PUBLIC)} kernel-verified public members loaded")
print(f"  kernel-verified     : {len(PRIOR_PUBLIC)}")
print(f"  library candidates  : {len(SZY)}")
_rej = [k for k, v in VER.items() if not v["verdict"].startswith("ADMISSIBLE")]
print(f"  rejected by the gate: {_rej}")

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

# THE CORRELATION SCREEN. Hash equality cannot see one configuration run in two kernels:
# the arrays differ by thread-level numerical noise. bag42 and lgb_raw sat in row 94 at
# Pearson 0.999970 and no check fired. Three lines, and it would have caught them.
Lz = np.column_stack([np.clip(np.log(np.clip(Poof[n], 1e-9, 1 - 1e-9)
                                     / (1 - np.clip(Poof[n], 1e-9, 1 - 1e-9))), -30, 30)
                      for n in names])
Cm = np.corrcoef(((Lz - Lz.mean(0)) / Lz.std(0)).T)
np.fill_diagonal(Cm, 0.0)
near = [(names[i], names[j], Cm[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
        if abs(Cm[i, j]) > 0.9999]
print(f"pairs above 0.9999: {len(near)}")
for a, b, r in near:
    print(f"  NEAR-DUPLICATE {a} and {b} at {r:+.6f}")
assert not near, "a near-duplicate pair is present, justify it or drop one"
print(f"removed from row 94's set: {[d[0] for d in DROPPED]}")
hi = sorted(((abs(Cm[i, j]), names[i], names[j])
             for i in range(len(names)) for j in range(i + 1, len(names))),
            reverse=True)[:3]
print("most collinear surviving pairs: "
      + ", ".join(f"{a}/{b} {r:.5f}" for r, a, b in hi))

  skip bandoof_band_mid.npy: shapes (115842,) / (50917,)
  skip bandoof_bandfm2.npy: shapes (158451,) / (70351,)


14 members selected from the last three sources: ['hbo_cat_strall_d8', 'hbo_kirill_o1', 'hbo_koda_exact_te', 'hbo_stringify_str3_d6', 'hbo_stringify_str3derived_d7', 'hbo_stringify_strall_d6', 'mk_xgb_v3', 'mk_lgb_v3', 'mk_cat_v3', 'fm_fmdeep', 'fm_fmnum', 'fm_fmplr', 'fm_fmpure', 'fm_fmwide']


67 srcL members carried, all matching manifest AUC


7 srcK members verified against their published per-fold AUCs


22 srcJ members loaded, AUC-checked, FOLD PROTOCOL UNVERIFIED
14 members loaded from srcO, srcQ and srcP
  fold protocol DOCUMENTED, NOT PROVEN. The offset on the submission is the check.
10 kernel-verified public members loaded
  kernel-verified     : 10
  library candidates  : 0
  rejected by the gate: ['lookup_srcA', 'spline_srcD']


175 vectors loaded, no exact duplicates


pairs above 0.9999: 0
removed from row 94's set: ['lgb_raw']
most collinear surviving pairs: realmlp/realmlp10 0.99948, rmlp_lat/rmlp_lat3 0.99942, cat42/cat7 0.99907


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE161 = ([IDX[n] for n, _ in BASE] + [IDX[n] for n in PRIOR_PUBLIC]
           + [IDX[n] for n in PRIOR_SZY + PRIOR_SZY2] + [IDX[n] for n in PRIOR_srcK]
           + [IDX[n] for n in PRIOR_srcJ])

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  hbo_cat_strall_d8   0.966587        0.980814       0.994444


  hbo_kirill_o1   0.968706        0.979938       0.979249


  hbo_koda_exact_te   0.968403        0.994908       0.989496


  hbo_stringify_str3_d6   0.965549        0.988666       0.984818


...

  fm_fmnum       0.967137        0.985228       0.979768


  fm_fmplr       0.967403        0.987108       0.982386


  fm_fmpure      0.964554        0.977905       0.984405


  fm_fmwide      0.964954        0.983074       0.984894



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
from sklearn.linear_model import LogisticRegression

# Row 153's exact configuration: the 175-member pool, pruned to 65 by mean absolute
# coefficient. Rebuilt here rather than assumed, so the base arm reproduces row 153.
ALL = BASE161 + [IDX[n] for n, _ in CAND]
print(f"{len(ALL)} members in the pool")


def fit_foldwise(cols, w=None):
    """Returns (per-fold AUC, oof scores, mean of the five fold-combiners on test)."""
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    for f in range(5):
        tr, va = folds != f, folds == f
        m = LogisticRegression(C=1.0, max_iter=4000).fit(
            Loof[np.ix_(tr, cols)], y[tr], sample_weight=None if w is None else w[tr])
        assert int(np.max(m.n_iter_)) < 4000, "combiner did not converge"
        oof[va] = m.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = m.decision_function(Ltest[:, cols])
        cf[f] = m.coef_[0]
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst.mean(axis=0), cf


def fit_fulldata(cols, w=None):
    """ONE combiner on all 691,369 rows, for the test vector only.

    Legitimate because no test row appears in the out-of-fold matrix, so nothing this
    combiner sees overlaps what it predicts. It is NOT used for CV, which would be
    scored on rows it was fitted on."""
    m = LogisticRegression(C=1.0, max_iter=4000).fit(Loof[:, cols], y, sample_weight=w)
    assert int(np.max(m.n_iter_)) < 4000
    return m.decision_function(Ltest[:, cols])


# The prune, from row 153's criterion.
_per_all, _, _coef = fit_foldwise(ALL)
_order = np.argsort(-np.abs(_coef.mean(axis=0)))
COLS = sorted(ALL[i] for i in _order[:65])
print(f"pruned to {len(COLS)} members")

# Importance weights from the adversarial model, w = p/(1-p), mean-normalised.
_adv = np.load(ROOT / "artifacts" / "oof" / "adversarial_train_prob.npy")
assert _adv.shape == (len(train),), _adv.shape
W = _adv / (1.0 - _adv)
W = W / W.mean()
print(f"adversarial weights: min {W.min():.3f} max {W.max():.3f} "
      f"p90/p10 {np.percentile(W, 90) / np.percentile(W, 10):.2f}")

per_A, test_A, _ = fit_foldwise(COLS)
per_C, test_C, _ = fit_foldwise(COLS, W)
test_B = fit_fulldata(COLS)
test_D = fit_fulldata(COLS, W)

ARMS_P = {"A_base": (per_A, test_A), "B_fulltest": (per_A, test_B),
          "C_advweight": (per_C, test_C), "D_both": (per_C, test_D)}
print()
for a, (p, _) in ARMS_P.items():
    print(f"  {a:12} CV {p.mean():.6f} +/- {p.std():.6f}")


175 members in the pool


pruned to 65 members
adversarial weights: min 0.207 max 4.646 p90/p10 1.92



  A_base       CV 0.969962 +/- 0.000379
  B_fulltest   CV 0.969962 +/- 0.000379
  C_advweight  CV 0.969959 +/- 0.000378
  D_both       CV 0.969959 +/- 0.000378


In [5]:
ROW153_CV = 0.969962
d0 = per_A.mean() - ROW153_CV
print(f"A reproduces row 153: {per_A.mean():.6f} vs {ROW153_CV:.6f}, delta {d0:+.2e}")
print(f"{'REPRODUCED' if abs(d0) < 1e-4 else 'FAILED, do not log this run'}\n")

d = per_C - per_A
sd = d.std(ddof=1)
print(f"importance weighting costs {d.mean():+.6f} on CV, "
      f"{int((d > 0).sum())}/5 folds, t(4)={d.mean() / (sd / np.sqrt(5)):.2f}")
print("EXPECTED. CV is measured on training rows and the weighting de-emphasises them.")
print("CV cannot rank these four arms: A and B share a CV, C and D share a CV, and the")
print("two changes are both aimed at the test vector. The leaderboard is the instrument.\n")

# How far apart are the four test vectors? If they agree to 0.9999 none of this matters.
import itertools
print("Spearman between the four test vectors:")
for a, b in itertools.combinations(ARMS_P, 2):
    r = pd.Series(ARMS_P[a][1]).corr(pd.Series(ARMS_P[b][1]), method="spearman")
    print(f"  {a:12} vs {b:12} {r:.6f}")

SUBD = ROOT / "submissions"
for a, (_, t) in ARMS_P.items():
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(t)) + 0.5) / len(t)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUBD / f"stack_proto_{a}.csv", index=False)
print(f"\nwrote four submissions, stack_proto_*.csv")
print("A is row 153 rebuilt and is the control; it should score 0.97103.")


A reproduces row 153: 0.969962 vs 0.969962, delta -4.77e-07
REPRODUCED

importance weighting costs -0.000003 on CV, 0/5 folds, t(4)=-3.33
EXPECTED. CV is measured on training rows and the weighting de-emphasises them.
CV cannot rank these four arms: A and B share a CV, C and D share a CV, and the
two changes are both aimed at the test vector. The leaderboard is the instrument.

Spearman between the four test vectors:
  A_base       vs B_fulltest   0.999999


  A_base       vs C_advweight  0.999992
  A_base       vs D_both       0.999996


  B_fulltest   vs C_advweight  0.999987
  B_fulltest   vs D_both       0.999995


  C_advweight  vs D_both       0.999997



wrote four submissions, stack_proto_*.csv
A is row 153 rebuilt and is the control; it should score 0.97103.


In [6]:
# This notebook varies the fitting protocol, not membership. The coefficient
# table belongs to the membership notebooks.
print("see the test-vector correlations above")


see the test-vector correlations above


In [7]:
print("Four submissions written. Membership is unchanged from row 153.")


Four submissions written. Membership is unchanged from row 153.
